# Feature Extraction

Loading the audio files and converting into spectrogram and uses train_test_split and saving the file in processed.

In [104]:
# imports
import librosa
import os
import numpy as np

File paths

In [105]:
WILDLIFE_PATH = "../datasets/processed/wildlife/"
THREAT_PATH = "../datasets/processed/threat/"
SAVE_PATH = "../datasets/processed/train_test/"

Define a function to do the conversion 
- The audio is loaded , trimmed or padded if greater than 5 or less than 5,
- converted into spectrogram and returned.

In [106]:
def specto_conv(path):
    arr = []
    for sample in os.listdir(path):
        if sample.endswith(".wav"):
            file = os.path.join(path,sample)
            y,sr = librosa.load(file,sr=22050)# 22050 is the standard value so that all sample is standardized.
            y = librosa.util.fix_length(y,size=int(5*22050)) # trimming of the audio and padding of the audio. hard coded!!
            spect = librosa.feature.melspectrogram(y=y,sr=sr,n_mels = 128)
            spectrogram = librosa.power_to_db(spect,ref=np.max)
            arr.append(spectrogram)
    return arr

Convert the wildlife and threat into spectrorams.

In [107]:
wildlife_spectro = specto_conv(WILDLIFE_PATH)
threat_spectro = specto_conv(THREAT_PATH)

In [108]:
X = wildlife_spectro + threat_spectro
Y = [0]*len(wildlife_spectro) + [1]*len(threat_spectro)

X = np.array(X)
Y = np.array(Y)

Combined both into a single file and created a label file according to the number of samples in wildlife and threat

In [109]:
X.shape

(840, 128, 216)

For CNN model we need (samples,height,width,channels) but we currently have (samples,height,width) to add that we use np.expand_dims

In [110]:
X = np.expand_dims(X, axis=1)

In [111]:
from sklearn.model_selection import train_test_split

X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.2,random_state=42,stratify=Y)

Splited the data , X and Y , into train test

In [112]:
print(f"X_train :{X_train.shape}")
print(f"X_test :{X_test.shape}")

print(f"Y_train :{Y_train.shape}")
print(f"Y_test :{Y_test.shape}")

X_train :(672, 1, 128, 216)
X_test :(168, 1, 128, 216)
Y_train :(672,)
Y_test :(168,)


In [113]:
np.save(os.path.join(SAVE_PATH,"X_train"),X_train)
np.save(os.path.join(SAVE_PATH,"X_test"),X_test)
np.save(os.path.join(SAVE_PATH,"Y_train"),Y_train)
np.save(os.path.join(SAVE_PATH,"Y_test"),Y_test)

Saved the the np array as such in npy file.